In [33]:
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import PandasTools, AllChem
import pandas as pd
import subprocess, re, shutil, os

In [ ]:
import sys

#仓库根：从当前目录往上找带 config.py 的目录（git clone 和直接下载 zip 都能用）
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import config

In [ ]:
WORK = config.XTB_WORK
XTB = config.XTB
TAR = config.PARENTS

def run_xtb(folder, xyz, *flags):
    with open(folder / "xtb.out", "w") as f:
        p = subprocess.run([str(XTB), xyz, *flags, "P", "4", "--json"],
                           cwd = folder, stdout = f, stderr = subprocess.STDOUT)
        return p.returncode

def read_energy(folder):
    text = (folder / "xtbout.json").read_text(encoding = "utf-8")
    return float(re.search(r'"total energy"\s*:\s*(-?[\d.eE+-]+)', text).group(1))

def select_best(md, mol):
    canon = Chem.MolToSmiles(mol)

    mol = Chem.AddHs(mol)
    para = AllChem.ETKDGv3()
    para.randomSeed = 7917
    fail1 = 0
    while fail1 < 5 and not AllChem.EmbedMultipleConfs(mol, numConfs = 10, params = para):
        para.useRandomCoords = True
        fail1 += 1
    if fail1 == 5:
        raise RuntimeError(f"{md.stem}: RDKit 重试 5 次仍无法生成 3D 构象")
    AllChem.UFFOptimizeMoleculeConfs(mol, maxIters = 1000, numThreads = 0)

    for i in mol.GetConformers():
        cid = i.GetId()
        Chem.MolToXYZFile(mol, str(md / f"conf_{cid:03d}.xyz"), confId = cid)
    
    fail2 = 0
    for i in sorted(md.glob("conf_*.xyz")):
        sub = md / i.stem
        sub.mkdir(exist_ok = True)
        i.replace(sub / "conf.xyz")
        rc = run_xtb(sub, "conf.xyz", "--gfn", "2", "--chrg", "0", "--uhf", "0", "--opt")
        if rc != 0:
            fail2 += 1
        if fail2 >= 3:
            raise RuntimeError(f"{md.stem}: 已有 {fail2} 个构象 xtb 优化失败，放弃该分子")
    
    success = [x for x in md.glob("conf_*") if (x / "xtbout.json").exists()]
    best = min(success, key = lambda x : read_energy(x))
    print(best.name, read_energy(best))
    shutil.copy2(md / best.name / "xtbopt.xyz", md / "parent.xyz")
    (md / "smiles.txt").write_text(canon, encoding = "utf-8")

In [35]:
file = config.DATA_TDT / "fixed.xlsx"
df = pd.read_excel(file)
PandasTools.AddMoleculeColumnToFrame(df, "SMILES", "ROMol", False)

In [36]:
bad = []
correct = [87, 118, 169, 434, 515, 683, 684, 693, 694, 774, 776, 900, 902, 972, 974, 1039, 1317]

for i in correct:
    md = Path(TAR / f"parent_{i:04d}")
    shutil.rmtree(md)
    md.mkdir(exist_ok = True)

    try:
        select_best(md, df.loc[i, "ROMol"])
    except Exception as err:
        print(f"[{i+1}/{len(df)}] {md.stem} 失败: {err}", flush = True)
        bad.append(md.stem)
        pd.DataFrame({"id": bad}).to_csv(md / "bad_ids.csv", index = False)

conf_008 -87.2894414
conf_005 -166.31148328
conf_006 -52.37903407
conf_006 -62.81002496
conf_000 -67.69698677
conf_009 -79.98348124
conf_007 -83.29285324
conf_009 -54.74301999
conf_008 -110.870375
conf_007 -48.99935791
conf_004 -27.10037403
conf_009 -70.3078829
conf_006 -43.21161003
conf_006 -100.22083585
conf_000 -60.70150629
conf_007 -73.27932719
conf_001 -138.09403101


In [37]:
correct = [87, 118, 169, 434, 515, 683, 684, 693, 694, 774, 776, 900, 902, 972, 974, 1039, 1317]

for i in correct:
    sub = TAR / f"parent_{i:04d}"
    os.rename(sub / "parent.xyz", sub / "conf.xyz")
    run_xtb(sub, "conf.xyz", "--gfn", "2", "--chrg", "0", "--uhf", "0", "--scc", "--wbo")

In [4]:
list = [23, 43, 643, 701, 1256, 1257, 1281, 1322, 1381]
for i in sorted(TAR.glob("parent_*")):
    if int(i.stem[-4 : ]) in list:
        shutil.rmtree(i)

In [5]:
md = config.PARENTS.parent / "Parents_"
md.mkdir(exist_ok = True)
for num, i in enumerate(sorted(TAR.glob("parent_*"))):
    os.rename(i, md / f"parent_{num:04d}")